# World Cup 2026 Predictor — Model Evaluation

Evaluates the calibrated LR+GBM ensemble on held-out international matches and backtests World Cup tournaments.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.model_selection import train_test_split

from app.services.ml.training_dataset import build_training_dataset, dataset_summary
from app.services.ml import model_loader

In [ ]:
X, y, weights, y_home, y_away = build_training_dataset(min_year=2000)
print(dataset_summary(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr = model_loader.load_ensemble_lr()
gbm = model_loader.load_ensemble_gbm()

lr_p = lr.predict_proba(X_test)
gbm_p = gbm.predict_proba(X_test)
ensemble_p = 0.45 * lr_p + 0.55 * gbm_p
pred = ensemble_p.argmax(axis=1)

print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Log loss: {log_loss(y_test, ensemble_p):.4f}")
brier = np.mean([brier_score_loss((y_test == c).astype(int), ensemble_p[:, c]) for c in range(3)])
print(f"Mean Brier (OVR): {brier:.4f}")

In [ ]:
# Calibration plot — home win probability vs observed frequency
home_probs = ensemble_p[:, 0]
bins = np.linspace(0, 1, 11)
bin_centers, observed = [], []
for i in range(len(bins) - 1):
    mask = (home_probs >= bins[i]) & (home_probs < bins[i + 1])
    if mask.sum() < 20:
        continue
    bin_centers.append((bins[i] + bins[i + 1]) / 2)
    observed.append((y_test[mask] == 0).mean())

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
plt.scatter(bin_centers, observed, s=60, c='#00c896', label='Home win')
plt.xlabel('Predicted P(home win)')
plt.ylabel('Observed frequency')
plt.title('Calibration — home win probability')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# World Cup backtest — matches from 2018 and 2022 (requires results in dataset)
import pandas as pd
from app.services.data.match_loader import load_results

df = load_results()
wc = df[df['tournament'].str.contains('World Cup', case=False, na=False)]
wc = wc[wc['date'].dt.year.isin([2018, 2022])]
print(f"World Cup 2018/2022 matches in dataset: {len(wc)}")
wc.groupby('date').size().head()